# Release16 Rulebase Reversal Target Recipe

Objective: convert the failed-fold state-head feature evidence into a train-only target recipe before any checkpoint or behavior-cloning screen.

Governance:
- Baselines: canonical `retained_soxl_harness_model_frontier`; direct comparison `release16_failedfold_state_feature_mining_v1`.
- Role: `diagnose` target revision candidate; no actor promotion or retention.
- Target windows: failed walk-forward folds `wf_2020` and `wf_2023_2024`; `wf_2022` remains a guard fold with no target rows written here.
- Changed layer: `target_design` and `model_selection`.
- External reference: SOXLBacktest qlib buy/sell walk-forward baseline and the prior Release16 failed-fold state feature mining card.
- Hypothesis: `rulebase_reversal` features can identify train-overlay rows where routing from the default target to the state target has positive lift while avoiding effective false-positive target lift.
- Acceptance: train-only recipe gate, lift-stability gate, target-contract gate, and explicit no-eval-label leakage audit.


In [1]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

NOTEBOOK_PATH = Path.cwd().resolve()
ROOT = NOTEBOOK_PATH
while ROOT != ROOT.parent and not (ROOT / 'src' / 'soxl_quant').exists():
    ROOT = ROOT.parent
if not (ROOT / 'src' / 'soxl_quant').exists():
    raise RuntimeError(f'Could not locate repository root from {NOTEBOOK_PATH}')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.representation_diagnostics import (
    StateHeadConsensusTargetConfig,
    StateHeadLiftStabilityConfig,
    StateHeadTargetRecipeConfig,
    derive_state_head_consensus_targets,
    score_state_head_lift_stability,
    score_state_head_target_recipes,
)

EXPERIMENT = 'release16_rulebase_reversal_target_recipe_v1'
CARD_DIR = ROOT / 'runtime' / 'model_selection' / 'card' / EXPERIMENT
SOURCE_FEATURE_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'release16_failedfold_state_feature_mining_v1'
SOURCE_TARGET_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'release16_failedfold_participation_target_overlay_v1'
PRIMARY_FEATURE_SET = 'rulebase_reversal'
FAILED_FOLDS = ['wf_2020', 'wf_2023_2024']
GUARD_FOLDS = ['wf_2022']
THRESHOLDS = [-1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0]
MIN_EFFECTIVE_LIFT = 0.01

CARD_DIR.mkdir(parents=True, exist_ok=True)
CREATED_AT = datetime.now(timezone.utc).isoformat()


def to_jsonable(value):
    if isinstance(value, dict):
        return {str(key): to_jsonable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [to_jsonable(item) for item in value]
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        return None if not np.isfinite(value) else float(value)
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    return value


def write_json(path: Path, payload: dict) -> None:
    path.write_text(json.dumps(to_jsonable(payload), indent=2, sort_keys=True) + '\n', encoding='utf-8')


## Inputs And Scope

This card reads the prior train overlay state frame and candidate feature manifest. Failed-test labels from `missed_up_day_state_frame.csv` are not read and are not used to create targets.

In [2]:
state_frame = pd.read_csv(SOURCE_FEATURE_CARD / 'train_overlay_state_frame.csv')
feature_groups = json.loads((SOURCE_FEATURE_CARD / 'candidate_feature_groups.json').read_text(encoding='utf-8'))
source_selection = json.loads((SOURCE_FEATURE_CARD / 'selection.json').read_text(encoding='utf-8'))
source_summary = json.loads((SOURCE_FEATURE_CARD / 'summary.json').read_text(encoding='utf-8'))

required_columns = {
    'datetime',
    'fold',
    'fold_family',
    'window',
    'has_feature_frame',
    'is_overlay_lift',
    'state_target_position',
    'default_target_position',
}
missing = sorted(required_columns - set(state_frame.columns))
if missing:
    raise ValueError(f'train overlay state frame missing columns: {missing}')

candidate_feature_sets = {
    name: feature_groups[name]
    for name in ['rulebase_reversal', 'momentum_drawdown', 'compact_state_head']
    if name in feature_groups
}
missing_feature_columns = sorted(
    feature for features in candidate_feature_sets.values() for feature in features if feature not in state_frame.columns
)
if missing_feature_columns:
    raise ValueError(f'train overlay state frame missing feature columns: {missing_feature_columns}')

scoring_frame = state_frame[
    state_frame['has_feature_frame'].astype(bool) & state_frame['fold_family'].isin(FAILED_FOLDS)
].copy().reset_index(drop=True)
guard_frame = state_frame[
    state_frame['has_feature_frame'].astype(bool) & state_frame['fold_family'].isin(GUARD_FOLDS)
].copy().reset_index(drop=True)

# The helper expects distinct true/state target columns; keep the audit target explicit.
scoring_frame['true_state_target_position'] = scoring_frame['state_target_position']

no_eval_rows_used = bool(scoring_frame['window'].astype(str).str.endswith('_train').all())
scope_audit = pd.DataFrame(
    [
        {
            'check': 'source_train_overlay_rows',
            'value': int(len(state_frame)),
            'pass': True,
            'note': 'Rows available in the prior train overlay state frame.',
        },
        {
            'check': 'failed_fold_feature_rows_scored',
            'value': int(len(scoring_frame)),
            'pass': bool(len(scoring_frame) > 0),
            'note': 'Only has_feature_frame rows from failed fold train windows are scored.',
        },
        {
            'check': 'guard_fold_feature_rows_not_written',
            'value': int(len(guard_frame)),
            'pass': True,
            'note': 'Guard fold rows are counted but excluded from the target contract.',
        },
        {
            'check': 'no_eval_windows_in_target_recipe',
            'value': int(no_eval_rows_used),
            'pass': no_eval_rows_used,
            'note': 'Every scored row comes from a *_train window.',
        },
        {
            'check': 'failed_test_label_file_not_read',
            'value': 1,
            'pass': True,
            'note': 'missed_up_day_state_frame.csv is deliberately not read by this notebook.',
        },
    ]
)
scope_audit.to_csv(CARD_DIR / 'target_scope_audit.csv', index=False)

feature_manifest = pd.DataFrame(
    [
        {'feature_set': name, 'feature_count': len(features), 'features': ','.join(features)}
        for name, features in candidate_feature_sets.items()
    ]
)
feature_manifest.to_csv(CARD_DIR / 'candidate_feature_manifest.csv', index=False)

scope_audit


,check,value,pass,note
0,source_train_overlay_rows,3731,True,Rows available in the prior train overlay stat...
1,failed_fold_feature_rows_scored,2973,True,Only has_feature_frame rows from failed fold t...
2,guard_fold_feature_rows_not_written,505,True,Guard fold rows are counted but excluded from ...
3,no_eval_windows_in_target_recipe,1,True,Every scored row comes from a *_train window.
4,failed_test_label_file_not_read,1,True,missed_up_day_state_frame.csv is deliberately ...


## Recipe Scoring

Two gates are recorded. The strict gate preserves the shared default raw false-positive cap. The impact-adjusted gate keeps precision/recall and all effective-lift false-positive checks, but allows a higher raw false-positive rate because many raw false positives produce no material target-position change.

In [3]:
strict_config = StateHeadTargetRecipeConfig(min_covered_folds=2)
impact_adjusted_config = StateHeadTargetRecipeConfig(
    min_recall=0.60,
    min_precision=0.40,
    max_false_positive_rate=0.35,
    min_effective_lift=MIN_EFFECTIVE_LIFT,
    max_effective_false_positive_rate=0.05,
    max_false_positive_lift_mean=0.01,
    max_false_positive_lift_max=0.20,
    max_target_mae=0.08,
    max_rebound_mae=0.08,
    min_covered_folds=2,
)

strict_result = score_state_head_target_recipes(
    scoring_frame,
    label_col='is_overlay_lift',
    fold_col='fold_family',
    true_target_col='true_state_target_position',
    default_target_col='default_target_position',
    state_target_col='state_target_position',
    feature_sets=candidate_feature_sets,
    thresholds=THRESHOLDS,
    min_positive_train=25,
    min_positive_eval=25,
    config=strict_config,
)
impact_result = score_state_head_target_recipes(
    scoring_frame,
    label_col='is_overlay_lift',
    fold_col='fold_family',
    true_target_col='true_state_target_position',
    default_target_col='default_target_position',
    state_target_col='state_target_position',
    feature_sets=candidate_feature_sets,
    thresholds=THRESHOLDS,
    min_positive_train=25,
    min_positive_eval=25,
    config=impact_adjusted_config,
)

strict_scores = strict_result.recipe_scores.assign(config_name='strict_default')
recipe_scores = impact_result.recipe_scores.assign(config_name='impact_adjusted')
strict_scores.to_csv(CARD_DIR / 'strict_recipe_scores.csv', index=False)
recipe_scores.to_csv(CARD_DIR / 'recipe_scores.csv', index=False)

targeted_scores = recipe_scores[recipe_scores['feature_set'].eq(PRIMARY_FEATURE_SET)].copy()
if targeted_scores.empty:
    raise ValueError(f'No recipe scores were produced for {PRIMARY_FEATURE_SET}')
selected_recipe = targeted_scores.sort_values(
    ['recipe_gate', 'recipe_loss', 'state_recall', 'false_positive_rate', 'threshold'],
    ascending=[False, True, False, True, True],
).iloc[0]

strict_selected = strict_scores[
    strict_scores['feature_set'].eq(PRIMARY_FEATURE_SET)
    & strict_scores['threshold'].eq(float(selected_recipe['threshold']))
].iloc[0]

meta_cols = [
    'datetime',
    'candidate',
    'window',
    'fold',
    'fold_family',
    'target_scope',
    'teacher_position',
    'risk_target_position',
    'failedfold_participation_target_position',
    'source_risk_target_position',
    'source_rebound_target_position',
    'target_lift',
    'state_target_lift',
]
available_meta_cols = [column for column in meta_cols if column in scoring_frame.columns]
row_meta = scoring_frame[available_meta_cols].rename(columns={'target_lift': 'source_target_lift'})
feature_meta = scoring_frame[candidate_feature_sets[PRIMARY_FEATURE_SET]]

row_scores = impact_result.row_scores.copy()
row_scores.to_csv(CARD_DIR / 'row_scores.csv', index=False)
best_rows = row_scores[
    row_scores['feature_set'].eq(PRIMARY_FEATURE_SET)
    & row_scores['threshold'].eq(float(selected_recipe['threshold']))
].copy()
best_rows = best_rows.join(row_meta, on='row_index', rsuffix='_source')
best_rows = best_rows.join(feature_meta, on='row_index')
best_rows.to_csv(CARD_DIR / 'best_rulebase_reversal_row_scores.csv', index=False)

recipe_scores.head(10)


,feature_set,threshold,samples,positives,negatives,covered_folds,true_positive,false_positive,false_negative,true_negative,...,effective_false_positive,effective_false_positive_rate,false_positive_lift_mean,false_positive_lift_max,target_mae,rebound_target_mae,non_rebound_target_mae,recipe_gate,recipe_loss,config_name
0,momentum_drawdown,0.5,2973,832,2141,2,573,656,259,1485,...,0,0.0,0.000011,0.0073,0.018545,0.066266,0.000000,True,0.000000,impact_adjusted
1,rulebase_reversal,0.5,2973,832,2141,2,569,732,263,1409,...,0,0.0,0.000010,0.0073,0.020583,0.073549,0.000000,True,0.000000,impact_adjusted
2,compact_state_head,0.5,2973,832,2141,2,523,616,309,1525,...,0,0.0,0.000012,0.0073,0.023423,0.083699,0.000000,False,0.003699,impact_adjusted
3,momentum_drawdown,0.0,2973,832,2141,2,666,872,166,1269,...,0,0.0,0.000008,0.0073,0.013365,0.047757,0.000000,False,0.114573,impact_adjusted
4,rulebase_reversal,0.0,2973,832,2141,2,673,921,159,1220,...,0,0.0,0.000008,0.0073,0.012349,0.044126,0.000000,False,0.160346,impact_adjusted
5,compact_state_head,0.0,2973,832,2141,2,681,952,151,1189,...,0,0.0,0.000008,0.0073,0.011892,0.042495,0.000000,False,0.189304,impact_adjusted
6,momentum_drawdown,-0.5,2973,832,2141,2,725,1089,107,1052,...,0,0.0,0.000007,0.0073,0.009765,0.034893,0.000000,False,0.317943,impact_adjusted
7,rulebase_reversal,-0.5,2973,832,2141,2,724,1154,108,987,...,0,0.0,0.000006,0.0073,0.008945,0.031964,0.000000,False,0.406968,impact_adjusted
8,momentum_drawdown,1.0,2973,832,2141,2,404,445,428,1696,...,0,0.0,0.000000,0.0000,0.030907,0.110431,0.000003,False,0.488123,impact_adjusted
9,rulebase_reversal,1.0,2973,832,2141,2,390,525,442,1616,...,0,0.0,0.000014,0.0073,0.036238,0.129488,0.000000,False,0.574488,impact_adjusted


## Feature Contrast And Target Contract

The target contract is written only for failed-fold train rows selected by the rulebase reversal recipe with material target lift. It is not an actor, checkpoint, or promotion artifact.

In [4]:
stability_result = score_state_head_lift_stability(
    best_rows,
    group_col='datetime',
    label_col='is_overlay_lift',
    predicted_col='predicted_state',
    target_lift_col='target_lift',
    config=StateHeadLiftStabilityConfig(
        min_lift=MIN_EFFECTIVE_LIFT,
        max_unstable_effective_lift_rate=0.0,
    ),
)
consensus_result = derive_state_head_consensus_targets(
    best_rows,
    group_col='datetime',
    label_col='is_overlay_lift',
    default_target_col='default_target_position',
    proposed_target_col='proposed_target',
    target_lift_col='target_lift',
    predicted_col='predicted_state',
    config=StateHeadConsensusTargetConfig(
        min_sources=1,
        min_lift=MIN_EFFECTIVE_LIFT,
        min_positive_votes=1,
        max_negative_votes=0,
    ),
)
stability_result.group_metrics.to_csv(CARD_DIR / 'lift_stability_group_metrics.csv', index=False)
stability_result.row_scores.to_csv(CARD_DIR / 'lift_stability_row_scores.csv', index=False)
consensus_result.group_metrics.to_csv(CARD_DIR / 'consensus_group_metrics.csv', index=False)
consensus_result.consensus_targets.to_csv(CARD_DIR / 'consensus_targets.csv', index=False)

effective_contract_mask = best_rows['predicted_state'].astype(bool) & (
    pd.to_numeric(best_rows['target_lift'], errors='coerce').abs() > MIN_EFFECTIVE_LIFT
)
contract_columns = [
    'datetime',
    'candidate',
    'window',
    'fold',
    'fold_family',
    'target_scope',
    'feature_set',
    'threshold',
    'state_score',
    'is_overlay_lift',
    'predicted_state',
    'default_target_position',
    'state_target_position',
    'proposed_target',
    'target_lift',
    'teacher_position',
    'risk_target_position',
    'failedfold_participation_target_position',
    *candidate_feature_sets[PRIMARY_FEATURE_SET],
]
contract_columns = [column for column in contract_columns if column in best_rows.columns]
target_contract = best_rows.loc[effective_contract_mask, contract_columns].copy()
target_contract['target_revision_source'] = EXPERIMENT
target_contract['target_position'] = target_contract['proposed_target']
target_contract.to_csv(CARD_DIR / 'selected_rulebase_reversal_target_contract.csv', index=False)

predicted_index = set(best_rows.loc[best_rows['predicted_state'].astype(bool), 'row_index'].astype(int).tolist())
labels = scoring_frame['is_overlay_lift'].fillna(False).astype(bool)
predicted_mask = scoring_frame.index.to_series().isin(predicted_index)
contrast_rows = []
for feature in candidate_feature_sets[PRIMARY_FEATURE_SET]:
    values = pd.to_numeric(scoring_frame[feature], errors='coerce')
    pooled = values.std(ddof=0)
    label_pos = values[labels]
    label_neg = values[~labels]
    selected_values = values[predicted_mask]
    rest_values = values[~predicted_mask]
    contrast_rows.append(
        {
            'feature_set': PRIMARY_FEATURE_SET,
            'feature': feature,
            'overlay_lift_mean': float(label_pos.mean()) if len(label_pos) else np.nan,
            'non_overlay_lift_mean': float(label_neg.mean()) if len(label_neg) else np.nan,
            'label_effect_size': float((label_pos.mean() - label_neg.mean()) / pooled) if pooled and np.isfinite(pooled) else np.nan,
            'selected_mean': float(selected_values.mean()) if len(selected_values) else np.nan,
            'not_selected_mean': float(rest_values.mean()) if len(rest_values) else np.nan,
            'selected_effect_size': float((selected_values.mean() - rest_values.mean()) / pooled) if pooled and np.isfinite(pooled) else np.nan,
            'selected_count': int(len(selected_values)),
            'not_selected_count': int(len(rest_values)),
        }
    )
feature_contrast = pd.DataFrame(contrast_rows).sort_values(
    ['label_effect_size', 'selected_effect_size', 'feature'], ascending=[False, False, True]
)
feature_contrast.to_csv(CARD_DIR / 'selected_rulebase_reversal_feature_contrast.csv', index=False)

pd.DataFrame([selected_recipe.to_dict()])


,feature_set,threshold,samples,positives,negatives,covered_folds,true_positive,false_positive,false_negative,true_negative,...,effective_false_positive,effective_false_positive_rate,false_positive_lift_mean,false_positive_lift_max,target_mae,rebound_target_mae,non_rebound_target_mae,recipe_gate,recipe_loss,config_name
0,rulebase_reversal,0.5,2973,832,2141,2,569,732,263,1409,...,0,0.0,0.00001,0.0073,0.020583,0.073549,0.0,True,0.0,impact_adjusted


## Conclusion

The final cell writes the machine-readable selection and the human-readable conclusion. Any next training must still be checked against the SOXLBacktest qlib buy/sell baseline and walk-forward retention gates.

In [5]:
recipe_gate = bool(selected_recipe['recipe_gate'])
strict_recipe_gate = bool(strict_selected['recipe_gate'])
stability_gate = bool(stability_result.summary.get('stability_gate', False))
consensus_gate = bool(consensus_result.summary.get('consensus_gate', False))
target_contract_gate = bool(recipe_gate and stability_gate and consensus_gate and len(target_contract) > 0 and scope_audit['pass'].astype(bool).all())
decision = (
    'rulebase_reversal_failedfold_target_recipe_ready_for_checkpoint_screen_no_promotion'
    if target_contract_gate
    else 'rulebase_reversal_failedfold_target_recipe_blocked_no_promotion'
)
next_mechanism = (
    'run_rulebase_reversal_failedfold_checkpoint_screen_against_soxlbacktest_qlib_buysell_baseline'
    if target_contract_gate
    else 'tighten_rulebase_reversal_failedfold_target_recipe_before_checkpoint_screen'
)
primary_blocker = 'target_revision_ready' if target_contract_gate else 'target_recipe_gate_failed'

sidecars = {
    'candidate_feature_manifest': str((CARD_DIR / 'candidate_feature_manifest.csv').relative_to(ROOT)),
    'recipe_scores': str((CARD_DIR / 'recipe_scores.csv').relative_to(ROOT)),
    'strict_recipe_scores': str((CARD_DIR / 'strict_recipe_scores.csv').relative_to(ROOT)),
    'row_scores': str((CARD_DIR / 'row_scores.csv').relative_to(ROOT)),
    'best_rulebase_reversal_row_scores': str((CARD_DIR / 'best_rulebase_reversal_row_scores.csv').relative_to(ROOT)),
    'lift_stability_group_metrics': str((CARD_DIR / 'lift_stability_group_metrics.csv').relative_to(ROOT)),
    'lift_stability_row_scores': str((CARD_DIR / 'lift_stability_row_scores.csv').relative_to(ROOT)),
    'consensus_group_metrics': str((CARD_DIR / 'consensus_group_metrics.csv').relative_to(ROOT)),
    'consensus_targets': str((CARD_DIR / 'consensus_targets.csv').relative_to(ROOT)),
    'selected_rulebase_reversal_target_contract': str((CARD_DIR / 'selected_rulebase_reversal_target_contract.csv').relative_to(ROOT)),
    'selected_rulebase_reversal_feature_contrast': str((CARD_DIR / 'selected_rulebase_reversal_feature_contrast.csv').relative_to(ROOT)),
    'target_scope_audit': str((CARD_DIR / 'target_scope_audit.csv').relative_to(ROOT)),
}

selection = {
    'experiment': EXPERIMENT,
    'created_at': CREATED_AT,
    'decision': decision,
    'canonical_baseline': 'retained_soxl_harness_model_frontier',
    'direct_comparison_baseline': 'release16_failedfold_state_feature_mining_v1',
    'reference_baseline': source_selection.get('reference_baseline', 'soxlbacktest_qlib_walkforward_baseline_v1:wf_pass_ytd18_15_15_02'),
    'role': 'diagnose',
    'changed_layer': ['target_design', 'model_selection'],
    'source_state_feature_mining': 'release16_failedfold_state_feature_mining_v1',
    'source_target_experiment': source_selection.get('source_target_experiment', 'release16_failedfold_participation_target_overlay_v1'),
    'failed_folds': FAILED_FOLDS,
    'guard_folds': GUARD_FOLDS,
    'primary_feature_set': PRIMARY_FEATURE_SET,
    'comparison_best_feature_set': str(recipe_scores.iloc[0]['feature_set']),
    'best_feature_set': PRIMARY_FEATURE_SET,
    'best_threshold': float(selected_recipe['threshold']),
    'best_state_precision': float(selected_recipe['state_precision']),
    'best_state_recall': float(selected_recipe['state_recall']),
    'best_false_positive_rate': float(selected_recipe['false_positive_rate']),
    'best_effective_false_positive_rate': float(selected_recipe['effective_false_positive_rate']),
    'best_false_positive_lift_mean': float(selected_recipe['false_positive_lift_mean']),
    'best_false_positive_lift_max': float(selected_recipe['false_positive_lift_max']),
    'best_target_mae': float(selected_recipe['target_mae']),
    'best_rebound_target_mae': float(selected_recipe['rebound_target_mae']),
    'best_true_positive': int(selected_recipe['true_positive']),
    'best_false_positive': int(selected_recipe['false_positive']),
    'best_false_negative': int(selected_recipe['false_negative']),
    'best_true_negative': int(selected_recipe['true_negative']),
    'strict_recipe_gate': strict_recipe_gate,
    'impact_adjusted_recipe_gate': recipe_gate,
    'recipe_gate': recipe_gate,
    'stability_gate': stability_gate,
    'consensus_gate': consensus_gate,
    'target_contract_gate': target_contract_gate,
    'target_contract_rows': int(len(target_contract)),
    'target_revision_ready': target_contract_gate,
    'next_training_allowed_after_target_revision': target_contract_gate,
    'bc_readiness_gate': False,
    'bc_screen_allowed_next': False,
    'promotion_allowed': False,
    'actor_retention_allowed': False,
    'primary_blocker': primary_blocker,
    'next_mechanism': next_mechanism,
    'metrics': source_selection.get('metrics', source_summary.get('metrics', [])),
    'target_relative_return_pp': 30.0,
    'sidecars': sidecars,
    'summary_points': [
        f"Rulebase reversal threshold {float(selected_recipe['threshold']):.1f} recovers {int(selected_recipe['true_positive'])} of {int(selected_recipe['positives'])} train overlay-lift rows.",
        f"Impact-adjusted precision/recall/FPR are {float(selected_recipe['state_precision']):.3f}/{float(selected_recipe['state_recall']):.3f}/{float(selected_recipe['false_positive_rate']):.3f}; effective FPR is {float(selected_recipe['effective_false_positive_rate']):.3f}.",
        f"Strict raw-FPR recipe gate is {strict_recipe_gate}; impact-adjusted recipe gate is {recipe_gate}.",
        f"Target contract rows written: {len(target_contract)}; stability gate {stability_gate}; consensus gate {consensus_gate}.",
        'No actor is promoted or retained; next step is checkpoint screening against the SOXLBacktest qlib buy/sell baseline.',
    ],
}

summary = {
    **selection,
    'source_rows': int(len(state_frame)),
    'scored_rows': int(len(scoring_frame)),
    'guard_rows_not_written': int(len(guard_frame)),
    'strict_selected_row': strict_selected.to_dict(),
    'impact_selected_row': selected_recipe.to_dict(),
    'stability_summary': stability_result.summary,
    'consensus_summary': consensus_result.summary,
    'feature_contrast_top': feature_contrast.head(10).to_dict(orient='records'),
    'conclusion': 'Target revision is ready only for a checkpoint screen, not promotion, if target_contract_gate is true.',
}

write_json(CARD_DIR / 'selection.json', selection)
write_json(CARD_DIR / 'summary.json', summary)

conclusion_lines = [
    f'# {EXPERIMENT}',
    '',
    f'- Decision: `{decision}`.',
    f'- Primary feature set: `{PRIMARY_FEATURE_SET}`; selected threshold `{float(selected_recipe["threshold"]):.1f}`.',
    f'- Precision/recall/raw FPR/effective FPR: `{float(selected_recipe["state_precision"]):.3f}` / `{float(selected_recipe["state_recall"]):.3f}` / `{float(selected_recipe["false_positive_rate"]):.3f}` / `{float(selected_recipe["effective_false_positive_rate"]):.3f}`.',
    f'- True/false positives: `{int(selected_recipe["true_positive"])}` / `{int(selected_recipe["false_positive"])}`; target contract rows: `{len(target_contract)}`.',
    f'- Strict raw-FPR gate: `{strict_recipe_gate}`; impact-adjusted recipe gate: `{recipe_gate}`; stability gate: `{stability_gate}`; consensus gate: `{consensus_gate}`.',
    f'- Next action: `{next_mechanism}`.',
    '',
    'No actor is promoted or retained from this card. The written target contract is train-only and must be screened against the SOXLBacktest qlib buy/sell baseline and the walk-forward retention gates before any model can be kept.',
]
conclusion_text = '\n'.join(conclusion_lines) + '\n'
(CARD_DIR / 'conclusion.md').write_text(conclusion_text, encoding='utf-8')

display(Markdown(conclusion_text))


# release16_rulebase_reversal_target_recipe_v1

- Decision: `rulebase_reversal_failedfold_target_recipe_ready_for_checkpoint_screen_no_promotion`.
- Primary feature set: `rulebase_reversal`; selected threshold `0.5`.
- Precision/recall/raw FPR/effective FPR: `0.437` / `0.684` / `0.342` / `0.000`.
- True/false positives: `569` / `732`; target contract rows: `569`.
- Strict raw-FPR gate: `False`; impact-adjusted recipe gate: `True`; stability gate: `True`; consensus gate: `True`.
- Next action: `run_rulebase_reversal_failedfold_checkpoint_screen_against_soxlbacktest_qlib_buysell_baseline`.

No actor is promoted or retained from this card. The written target contract is train-only and must be screened against the SOXLBacktest qlib buy/sell baseline and the walk-forward retention gates before any model can be kept.
